In [0]:
%sql

DROP TABLE IF EXISTS observatorio_dev.gold.fact_generacion_real;
DROP TABLE IF EXISTS observatorio_dev.gold.fact_disponibilidad_planta;
DROP TABLE IF EXISTS observatorio_dev.gold.fact_demanda_real;
DROP TABLE IF EXISTS observatorio_dev.gold.fact_precio_bolsa;
DROP TABLE IF EXISTS observatorio_dev.gold.fact_energia_embalsada_planta;

DROP TABLE IF EXISTS observatorio_dev.gold.bridge_planta_embalse;

DROP TABLE IF EXISTS observatorio_dev.gold.dim_agente;
DROP TABLE IF EXISTS observatorio_dev.gold.dim_planta;
DROP TABLE IF EXISTS observatorio_dev.gold.dim_embalse;
DROP TABLE IF EXISTS observatorio_dev.gold.dim_periodo;
DROP TABLE IF EXISTS observatorio_dev.gold.dim_fecha;

# DIMENSIONS

### Dim Fecha

In [0]:
%sql

CREATE TABLE observatorio_dev.gold.dim_fecha (
    fecha_key INT NOT NULL
        COMMENT 'Clave determinística de fecha en formato YYYYMMDD',

    fecha DATE NOT NULL
        COMMENT 'Fecha calendario',

    anio SMALLINT NOT NULL,
    semestre TINYINT NOT NULL,
    trimestre TINYINT NOT NULL,

    mes_numero TINYINT NOT NULL,
    mes_nombre STRING NOT NULL,
    mes_nombre_corto STRING NOT NULL,

    anio_mes INT NOT NULL
        COMMENT 'Identificador en formato YYYYMM',

    anio_mes_nombre STRING NOT NULL,

    semana_anio TINYINT NOT NULL,
    dia_anio SMALLINT NOT NULL,
    dia_mes TINYINT NOT NULL,

    dia_semana_numero TINYINT NOT NULL
        COMMENT 'Día ISO: lunes 1 a domingo 7',

    dia_semana_nombre STRING NOT NULL,

    es_fin_semana BOOLEAN NOT NULL,
    es_inicio_mes BOOLEAN NOT NULL,
    es_fin_mes BOOLEAN NOT NULL,

    fecha_creacion TIMESTAMP NOT NULL,
    fecha_actualizacion TIMESTAMP NOT NULL
)
USING DELTA
COMMENT 'Dimensión calendario conformada del observatorio energético'
TBLPROPERTIES (
    'delta.enableChangeDataFeed' = 'true',
    'quality' = 'gold'
);

### Dim Periodo

In [0]:
%sql

CREATE TABLE observatorio_dev.gold.dim_periodo (
    periodo_key TINYINT NOT NULL
        COMMENT 'Clave del periodo horario entre 1 y 24',

    numero_periodo TINYINT NOT NULL,
    hora_inicio TINYINT NOT NULL,
    hora_fin TINYINT NOT NULL,

    hora_inicio_etiqueta STRING NOT NULL,
    hora_fin_etiqueta STRING NOT NULL,

    periodo_etiqueta STRING NOT NULL,
    rango_horario STRING NOT NULL,

    fecha_creacion TIMESTAMP NOT NULL,
    fecha_actualizacion TIMESTAMP NOT NULL
)
USING DELTA
COMMENT 'Dimensión conformada de los 24 periodos horarios'
TBLPROPERTIES (
    'delta.enableChangeDataFeed' = 'true',
    'quality' = 'gold'
);

### Dim Agente

In [0]:
%sql

CREATE TABLE observatorio_dev.gold.dim_agente (
    agente_key BIGINT
        GENERATED ALWAYS AS IDENTITY
        COMMENT 'Clave sustituta de la versión histórica del agente',

    codigo_agente STRING NOT NULL
        COMMENT 'Código natural del agente',

    nombre_agente STRING NOT NULL,
    nombre_agente_normalizado STRING NOT NULL,

    actividad_agente STRING NOT NULL,
    actividad_normalizada STRING NOT NULL,

    numero_version INT NOT NULL,

    fecha_inicio DATE NOT NULL
        COMMENT 'Inicio inclusivo de vigencia',

    fecha_fin DATE NOT NULL
        COMMENT 'Fin inclusivo de vigencia',

    es_actual BOOLEAN NOT NULL,

    fecha_creacion TIMESTAMP NOT NULL,
    fecha_actualizacion TIMESTAMP NOT NULL
)
USING DELTA
COMMENT 'Dimensión SCD Tipo 2 de agentes del mercado energético'
TBLPROPERTIES (
    'delta.enableChangeDataFeed' = 'true',
    'quality' = 'gold'
);

### Dim Planta

In [0]:
%sql

CREATE TABLE observatorio_dev.gold.dim_planta (
    planta_key BIGINT
        GENERATED ALWAYS AS IDENTITY
        COMMENT 'Clave sustituta estable de la planta o recurso',

    codigo_planta STRING NOT NULL
        COMMENT 'Código natural de planta o recurso',

    nombre_planta STRING NOT NULL
        COMMENT 'Nombre oficial o nombre provisional del recurso inferido',

    codigo_sic_agente STRING
        COMMENT 'Código SIC del agente asociado en el maestro vigente',

    cap_efectiva_neta DECIMAL(24,6)
        COMMENT 'Capacidad efectiva neta reportada por la fuente',

    fpo DATE
        COMMENT 'Fecha de puesta en operación',

    codigo_sub_area_operativa STRING,
    codigo_area_operativa STRING,

    tipo_despacho_recurso STRING,
    tipo_clasificacion STRING,
    tipo_generacion STRING,

    es_registro_inferido BOOLEAN NOT NULL
        COMMENT 'Indica si el recurso fue creado desde una fuente operativa por ausencia en el maestro',

    origen_registro STRING NOT NULL
        COMMENT 'Fuente que originó inicialmente el miembro',

    esta_en_maestro_actual BOOLEAN NOT NULL
        COMMENT 'Indica si el código existe actualmente en silver.plantas',

    fecha_primera_observacion DATE
        COMMENT 'Primera fecha observada en maestro o fuentes operativas',

    fecha_ultima_observacion DATE
        COMMENT 'Última fecha observada en maestro o fuentes operativas',

    fecha_creacion TIMESTAMP NOT NULL,
    fecha_actualizacion TIMESTAMP NOT NULL
)
USING DELTA
COMMENT 'Dimensión vigente Tipo 1 de plantas y recursos, incluyendo miembros inferidos'
TBLPROPERTIES (
    'delta.enableChangeDataFeed' = 'true',
    'quality' = 'gold'
);

### Dim Embalse

In [0]:
%sql

CREATE TABLE observatorio_dev.gold.dim_embalse (
    embalse_key BIGINT
        GENERATED ALWAYS AS IDENTITY
        COMMENT 'Clave sustituta estable del embalse',

    codigo_embalse STRING NOT NULL,
    nombre_embalse STRING NOT NULL,
    nombre_embalse_normalizado STRING NOT NULL,

    latitud DECIMAL(10,7),
    longitud DECIMAL(11,7),

    tipo_coordenada STRING,
    fuente_coordenada STRING,
    estado_geocodificacion STRING,
    consulta_geocodificacion STRING,

    coordenadas_validas BOOLEAN NOT NULL,
    requiere_revision_manual BOOLEAN NOT NULL,

    source_file_name STRING,
    source_file_path STRING,
    ingestion_timestamp TIMESTAMP,
    silver_load_date DATE,

    fecha_creacion TIMESTAMP NOT NULL,
    fecha_actualizacion TIMESTAMP NOT NULL
)
USING DELTA
COMMENT 'Dimensión Tipo 1 de embalses, coordenadas y trazabilidad geográfica'
TBLPROPERTIES (
    'delta.enableChangeDataFeed' = 'true',
    'quality' = 'gold'
);

# Bridge Table

###Planta-Embalse

In [0]:
%sql

CREATE TABLE observatorio_dev.gold.bridge_planta_embalse (
    planta_embalse_key STRING NOT NULL
        COMMENT 'Clave SHA-256 de la relación planta-embalse',

    planta_key BIGINT NOT NULL,
    embalse_key BIGINT NOT NULL,

    codigo_planta STRING NOT NULL,
    codigo_embalse STRING NOT NULL,

    region STRING,

    nombre_planta_fuente STRING,
    nombre_reservorio_fuente STRING,

    tipo_relacion STRING,
    es_principal BOOLEAN NOT NULL,
    permite_atribucion BOOLEAN NOT NULL,

    fuente_relacion STRING,
    estado_validacion STRING,

    valido_desde DATE,
    valido_hasta DATE,

    cantidad_embalses_planta INT NOT NULL,

    es_relacion_unica BOOLEAN NOT NULL
        COMMENT 'Indica si la planta está vinculada con un solo embalse',

    requiere_revision_manual BOOLEAN NOT NULL,

    fecha_creacion TIMESTAMP NOT NULL,
    fecha_actualizacion TIMESTAMP NOT NULL
)
USING DELTA
COMMENT 'Puente gobernado entre plantas y embalses'
TBLPROPERTIES (
    'delta.enableChangeDataFeed' = 'true',
    'quality' = 'gold'
);

# FACTS

### Fact Generacion Real

In [0]:
%sql

CREATE TABLE observatorio_dev.gold.fact_generacion_real (
    generacion_key STRING NOT NULL
        COMMENT 'Clave determinística de la medición consolidada',

    fecha_key INT NOT NULL,
    periodo_key TINYINT NOT NULL,

    planta_key BIGINT NOT NULL,
    agente_key BIGINT NOT NULL,

    fecha_hora TIMESTAMP NOT NULL,

    generacion_real_kwh DECIMAL(24,6) NOT NULL,

    version_seleccionada STRING NOT NULL,
    prioridad_version INT NOT NULL,

    planta_provenia_de_maestro BOOLEAN NOT NULL
        COMMENT 'Indicador de calidad heredado de Silver antes de resolver miembros inferidos',

    agente_encontrado_silver BOOLEAN NOT NULL
        COMMENT 'Indica si el agente tenía correspondencia en Silver',

    fecha_creacion TIMESTAMP NOT NULL,
    fecha_actualizacion TIMESTAMP NOT NULL
)
USING DELTA
COMMENT 'Generación real horaria consolidada por planta y agente'
TBLPROPERTIES (
    'delta.enableChangeDataFeed' = 'true',
    'quality' = 'gold'
);

### Fact Disponibilidad Planta

In [0]:
%sql

CREATE TABLE observatorio_dev.gold.fact_disponibilidad_planta (
    disponibilidad_key STRING NOT NULL,

    fecha_key INT NOT NULL,
    periodo_key TINYINT NOT NULL,
    planta_key BIGINT NOT NULL,

    fecha_hora TIMESTAMP NOT NULL,

    disponibilidad_real_kwh DECIMAL(24,6) NOT NULL,

    version_seleccionada STRING NOT NULL,
    prioridad_version INT NOT NULL,

    planta_provenia_de_maestro BOOLEAN NOT NULL,

    fecha_creacion TIMESTAMP NOT NULL,
    fecha_actualizacion TIMESTAMP NOT NULL
)
USING DELTA
COMMENT 'Disponibilidad real horaria consolidada por planta o recurso'
TBLPROPERTIES (
    'delta.enableChangeDataFeed' = 'true',
    'quality' = 'gold'
);

### Fact Demanda Real

In [0]:
%sql

CREATE TABLE observatorio_dev.gold.fact_demanda_real (
    demanda_key STRING NOT NULL,

    fecha_key INT NOT NULL,
    periodo_key TINYINT NOT NULL,
    agente_key BIGINT NOT NULL,

    fecha_hora TIMESTAMP NOT NULL,
    tipo_mercado STRING NOT NULL,

    demanda_real_kwh DECIMAL(24,6) NOT NULL,
    es_demanda_cero BOOLEAN NOT NULL,

    version_seleccionada STRING NOT NULL,
    prioridad_version INT NOT NULL,

    agente_encontrado_silver BOOLEAN NOT NULL,

    fecha_creacion TIMESTAMP NOT NULL,
    fecha_actualizacion TIMESTAMP NOT NULL
)
USING DELTA
COMMENT 'Demanda real horaria consolidada por agente y mercado'
TBLPROPERTIES (
    'delta.enableChangeDataFeed' = 'true',
    'quality' = 'gold'
);

### Fact Precio Bolsa

In [0]:
%sql

CREATE TABLE observatorio_dev.gold.fact_precio_bolsa (
    precio_bolsa_key STRING NOT NULL,

    fecha_key INT NOT NULL,
    periodo_key TINYINT NOT NULL,

    fecha_hora TIMESTAMP NOT NULL,

    precio_bolsa_internacional_cop_kwh DECIMAL(24,6),
    precio_bolsa_nacional_cop_kwh DECIMAL(24,6),
    precio_bolsa_tie_cop_kwh DECIMAL(24,6),

    version_pb_int STRING,
    prioridad_pb_int INT,

    version_pb_nal STRING,
    prioridad_pb_nal INT,

    version_pb_tie STRING,
    prioridad_pb_tie INT,

    fecha_creacion TIMESTAMP NOT NULL,
    fecha_actualizacion TIMESTAMP NOT NULL
)
USING DELTA
COMMENT 'Precios horarios PB_INT, PB_NAL y PB_TIE consolidados en formato ancho'
TBLPROPERTIES (
    'delta.enableChangeDataFeed' = 'true',
    'quality' = 'gold'
);

### Fact Energia embalsamada Planta

In [0]:
%sql

CREATE TABLE observatorio_dev.gold.fact_energia_embalsada_planta (
    energia_embalsada_key STRING NOT NULL,

    fecha_key INT NOT NULL,
    planta_key BIGINT NOT NULL,

    fecha_medicion DATE NOT NULL,

    energia_embalsada_kwh DECIMAL(24,6) NOT NULL,
    es_valor_cero BOOLEAN NOT NULL,

    version_seleccionada STRING NOT NULL,
    prioridad_version INT NOT NULL,

    planta_provenia_de_maestro BOOLEAN NOT NULL,

    fecha_creacion TIMESTAMP NOT NULL,
    fecha_actualizacion TIMESTAMP NOT NULL
)
USING DELTA
COMMENT 'Snapshot diario de energía embalsada reportada por planta'
TBLPROPERTIES (
    'delta.enableChangeDataFeed' = 'true',
    'quality' = 'gold'
);

# Validar objetos creados

In [0]:
%sql

SHOW TABLES IN observatorio_dev.gold;